# Real-Time Color Recognition Using OpenCV

This project detects and recognizes colored objects using OpenCV.

Pipeline:
- Convert image from BGR to HSV
- Apply color thresholding
- Detect object contours
- Draw bounding boxes and labels

## Download & Import Libraries

In [1]:
!pip install opencv-python numpy

import cv2
import numpy as np

## Define HSV Color Ranges

HSV is used because it separates color information from brightness, making color detection more reliable under different lighting conditions.

In [2]:
colors = {

    "Red": [
        (np.array([0, 160, 80]), np.array([10, 255, 255])),
        (np.array([170, 160, 80]), np.array([180, 255, 255]))
    ],

    "Green": [
        (np.array([35, 80, 50]), np.array([85, 255, 255]))
    ],

    "Blue": [
        (np.array([90, 80, 50]), np.array([130, 255, 255]))
    ],

    "Yellow": [
        (np.array([20, 100, 100]), np.array([35, 255, 255]))
    ]
}

## Initialize Webcam

The webcam provides continuous frames that will be processed in real time.

In [3]:
cap = cv2.VideoCapture(0)

## Real-Time Color Detection

Each frame is:
1. Converted to HSV.
2. Segmented based on color ranges.
3. Processed to remove noise.
4. Analyzed using contours.

In [ ]:
while True:

    ret, frame = cap.read()

    if not ret:
        break


    # Convert BGR image to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)


    for color_name, ranges in colors.items():

        # Create empty mask
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)


        # Combine HSV ranges
        for lower, upper in ranges:
            mask += cv2.inRange(hsv, lower, upper)


        # Remove noise
        kernel = np.ones((5,5), np.uint8)

        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_OPEN,
            kernel
        )

        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            kernel
        )


        # Find object contours
        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )


        for contour in contours:

            area = cv2.contourArea(contour)


            if area > 500:

                # Bounding box
                x, y, w, h = cv2.boundingRect(contour)


                # Draw rectangle
                cv2.rectangle(
                    frame,
                    (x,y),
                    (x+w,y+h),
                    (0,255,0),
                    2
                )


                # Calculate center
                center_x = x + w//2
                center_y = y + h//2


                cv2.circle(
                    frame,
                    (center_x, center_y),
                    5,
                    (255,0,0),
                    -1
                )


                # Display color name
                cv2.putText(
                    frame,
                    color_name,
                    (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0,255,0),
                    2
                )


                # Display center coordinates
                cv2.putText(
                    frame,
                    f"Center: {center_x},{center_y}",
                    (x,y+h+25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255,255,255),
                    2
                )


    # Show final result
    cv2.imshow(
        "Color Recognition",
        frame
    )


    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

## Release Resources

Close the webcam and OpenCV windows after finishing.

In [5]:
cap.release()
cv2.destroyAllWindows()